# 15 — Cost and timing
Simulation, features, training (from notebook 06 metadata) and inference per window.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

In [ ]:
t = time.time(); d = da.make_windows(sim.prior_draws(10_000, 15100), 15100); t_sim = (time.time() - t) / 10_000
x = d["x"][:720]
t = time.time(); inf.sample_within_prior(post, x[:1], 1000); t_one = time.time() - t
t = time.time(); inf.sample_within_prior(post, x, 1000); t_batch = (time.time() - t) / len(x)
cost = dict(sim_and_features_per_window_s=t_sim, sbi_single_window_s=t_one, sbi_batched_per_window_s=t_batch)
display(pd.Series(cost)); da.save_json(R / "15_cost.json", cost)

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).